In [1]:
# Verif GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

CUDA available: True
GPU: Tesla T4


In [2]:
!pip install unsloth trl peft accelerate bitsandbytes datasets -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 123.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 107.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 113.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 111.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [3]:
# Upload dataset_v4_train.jsonl et dataset_v4_val.jsonl
from google.colab import files
files.upload()

Saving dataset_merged_val.jsonl to dataset_merged_val.jsonl
Saving dataset_merged_train.jsonl to dataset_merged_train.jsonl


{'dataset_merged_val.jsonl': b'{"input": "un pouf devant un cube a 0.7 m avec un fauteuil sous le cube", "config": "basic", "output": {"objets": ["pouf", "fauteuil", "cube"], "root": "pouf", "relations": [{"type": "under", "subject": "fauteuil", "object": "cube"}, {"type": "in_front_of", "subject": "pouf", "object": "cube", "distance": 0.7}], "orientations": []}}\n{"input": "un bureau a gauche d\'un pouf qui est devant une banane", "config": "basic", "output": {"objets": ["bureau", "pouf", "banane"], "root": "bureau", "relations": [{"type": "in_front_of", "subject": "pouf", "object": "banane"}, {"type": "left_of", "subject": "bureau", "object": "pouf"}], "orientations": []}}\n{"input": "A washing machine with a bottle on top, a refrigerator against it, and a plant to its right.", "config": "anglais", "output": {"objets": ["washing_machine", "refrigerator", "plant", "bottle"], "root": "washing_machine", "relations": [{"type": "on", "subject": "bottle", "object": "washing_machine"}, {"ty

In [ ]:
import json
from datasets import Dataset

SYSTEM_PROMPT = (
    'Tu es un assistant qui extrait les objets, relations spatiales, distances et orientations '
    "d'une description de scene. Reponds uniquement en JSON valide.\n"
    'Format : {"objets": [...], '
    '"relations": [{"type": "...", "subject": "...", "object": "...", "distance": <float optionnel>}], '
    '"orientations": [{"id": "...", "turn": "..."}]}\n'
    'Types de relations : on, under, left_of, right_of, in_front_of, behind, against, inside\n'
    "Champ 'distance' (en metres) autorise UNIQUEMENT sur : left_of, right_of, in_front_of, behind\n"
    'Turns valides : tip_left, tip_right, tip_forward, tip_backward, upside_down, turn_left, turn_right, turn_around\n'
    "Convention : premiere instance = 'banane', deuxieme = 'banane_2'.\n"
    'Si pas de distance mentionnee : ne pas inclure le champ. Si tout est debout : orientations = [].'
)

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

def format_prompt(example):
    output_str = json.dumps(example['output'], ensure_ascii=False)
    return (
        f'<|system|>\n{SYSTEM_PROMPT}<|end|>\n'
        f"<|user|>\n{example['input']}<|end|>\n"
        f'<|assistant|>\n{output_str}<|end|>'
    )

train_raw = load_jsonl('dataset_v4_raw_train.jsonl')
val_raw   = load_jsonl('dataset_v4_raw_val.jsonl')

train_ds = Dataset.from_dict({'text': [format_prompt(ex) for ex in train_raw]})
val_ds   = Dataset.from_dict({'text': [format_prompt(ex) for ex in val_raw]})

print(f'Train : {len(train_ds)} | Val : {len(val_ds)}')
print('Exemple :')
print(train_ds[0]['text'][:300])

In [8]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Phi-3-mini-4k-instruct-bnb-4bit',
    max_seq_length=768,
    dtype=None,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.4.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [9]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

Unsloth 2026.4.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field='text',
    max_seq_length=768,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=50,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        output_dir='phi3_finetuned_v4',
        report_to='none',
    ),
)

trainer.train()

In [11]:
# Test rapide
FastLanguageModel.for_inference(model)

test_prompt = 'je veux un frigo a cote d une poubelle, derriere cette poubelle un lave linge'
inputs = tokenizer(
    f'<|system|>\n{SYSTEM_PROMPT}<|end|>\n<|user|>\n{test_prompt}<|end|>\n<|assistant|>\n',
    return_tensors='pt'
).to('cuda')

out = model.generate(**inputs, max_new_tokens=256, temperature=0.0, do_sample=False)
print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

{"objets": ["poubelle", "frigo", "lave_linge"], "root": "frigo", "relations": [{"type": "behind", "subject": "lave_linge", "object": "poubelle"}, {"type": "left_of", "subject": "frigo", "object": "poubelle"}]}


In [ ]:
# Pre-installation llama.cpp pendant que l'internet est disponible
import urllib.request, subprocess, os, glob, shutil

try:
    urllib.request.urlopen("https://github.com", timeout=5)
    print("Internet OK")
except Exception as e:
    raise RuntimeError("Pas d'internet - verifie dans Runtime > Change runtime type")

llama_dir = "/root/.unsloth/llama.cpp"
quantizer = glob.glob(f"{llama_dir}/llama-quantize") + glob.glob(f"{llama_dir}/build/**/llama-quantize", recursive=True)

if not quantizer:
    print("Installation llama.cpp...")
    if not os.path.exists(llama_dir):
        subprocess.run(["git", "clone", "https://github.com/ggerganov/llama.cpp", llama_dir], check=True)
    build_dir = f"{llama_dir}/build"
    os.makedirs(build_dir, exist_ok=True)
    subprocess.run(["cmake", "..", "-DCMAKE_BUILD_TYPE=Release", "-DLLAMA_CURL=OFF"], cwd=build_dir, check=True)
    subprocess.run(["cmake", "--build", ".", "--config", "Release", "-j4", "--target", "llama-quantize"], cwd=build_dir, timeout=None)
    bins = glob.glob(f"{build_dir}/**/llama-quantize", recursive=True)
    if bins:
        shutil.copy(bins[0], f"{llama_dir}/llama-quantize")
        os.chmod(f"{llama_dir}/llama-quantize", 0o755)
    print("llama.cpp installe")
else:
    print("llama.cpp deja present")

In [12]:
# Export GGUF (pour Ollama sur ton Mac)
model.save_pretrained_gguf('phi3_finetuned_gguf', tokenizer, quantization_method='q4_k_m')

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in phi3_finetuned_gguf/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in phi3_finetuned_gguf.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.65G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [06:18<00:00, 189.19s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:18<00:00, 99.04s/it]


Unsloth: Merge process complete. Saved to `/content/phi3_finetuned_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['phi3_finetuned_gguf_gguf/phi-3-mini-4k-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['phi3_finetuned_gguf_gguf/phi-3-mini-4k-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model phi3_finetuned_gguf_gguf/phi-3-mini-4k-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to phi3_finetuned_gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f phi3_finetuned_gguf_gguf/Modelfile


{'save_directory': 'phi3_finetuned_gguf',
 'gguf_directory': 'phi3_finetuned_gguf_gguf',
 'gguf_files': ['phi3_finetuned_gguf_gguf/phi-3-mini-4k-instruct.Q4_K_M.gguf'],
 'modelfile_location': 'phi3_finetuned_gguf_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [15]:
import os
from google.colab import files

gguf_files = [f for f in os.listdir('phi3_finetuned_gguf_gguf') if f.endswith('.gguf')]
print(f'Fichier GGUF : {gguf_files}')
files.download(os.path.join('phi3_finetuned_gguf_gguf', gguf_files[0]))

Fichier GGUF : ['phi-3-mini-4k-instruct.Q4_K_M.gguf']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

gguf_src = 'phi3_finetuned_gguf_gguf/phi-3-mini-4k-instruct.Q4_K_M.gguf'
modelfile_src = 'phi3_finetuned_gguf_gguf/Modelfile'
drive_dir = '/content/drive/MyDrive/'

shutil.copy(gguf_src, os.path.join(drive_dir, 'phi-3-mini-4k-instruct.Q4_K_M.gguf'))
shutil.copy(modelfile_src, os.path.join(drive_dir, 'Modelfile_phi3'))
print("Copie terminee !")

Mounted at /content/drive
Copie terminee !
